# 🌸 Image Classification with Transfer Learning using PyTorch

**Author**: Sadiraja  
**Repository**: [Image-Classifier-with-Transfer-Learning](https://github.com/Sadiraja/Image-Classifier-with-Transfer-Learning)  
**Model**: Pretrained MobileNetV2 fine-tuned on 5 Flower Classes  
**Dataset**: Google / TensorFlow Flowers (Daisy, Dandelion, Roses, Sunflowers, Tulips)

---

## 📌 Project Overview
Transfer learning allows leveraging representations learned from large-scale visual benchmarks (ImageNet) to achieve high predictive accuracy on domain-specific datasets with reduced training times and small sample sizes.

In this notebook, we present the complete computer vision workflow:
1. **Dataset Organization**: Stratified partitioning into Train (70%), Validation (15%), and Test (15%) splits.
2. **Data Augmentation**: Robust geometric and color augmentations (`RandomResizedCrop`, `RandomRotation`, `RandomHorizontalFlip`, `ColorJitter`) to avoid overfitting.
3. **Two-Stage Transfer Learning**:
   - *Stage 1 (Feature Extraction)*: Frozen backbone, training the customized classification head.
   - *Stage 2 (Fine-Tuning)*: Top convolutional layers unfrozen with Cosine Annealing learning rate schedule.
4. **Performance Evaluation**: Comprehensive metrics including accuracy, precision, recall, F1-scores, and a normalized confusion matrix heatmap.
5. **Interactive Prediction**: Classifying novel images with confidence probability distributions.



In [ ]:
import os
import sys
import json
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report, confusion_matrix

# Add project root directory to path
sys.path.insert(0, os.path.abspath('..'))

from models.classifier import get_model, load_checkpoint
from src.dataset import get_transforms, create_dataloaders
from src.predict import predict_image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch Version: {torch.__version__}')
print(f'Active Device: {device}')



## 📂 1. Dataset & Data Augmentation Pipeline

The dataset consists of 3,670 high-resolution floral photographs across 5 categories:
- **Daisy**: 633 images (Train: 443, Val: 94, Test: 96)
- **Dandelion**: 898 images (Train: 628, Val: 134, Test: 136)
- **Roses**: 641 images (Train: 448, Val: 96, Test: 97)
- **Sunflowers**: 699 images (Train: 489, Val: 104, Test: 106)
- **Tulips**: 799 images (Train: 559, Val: 119, Test: 121)

Data augmentations are applied dynamically during training to make the model invariant to scale, rotation, flipping, and lighting variances.



In [ ]:
dataset_dir = os.path.join('..', 'dataset')
train_loader, val_loader, test_loader, class_names = create_dataloaders(
    dataset_dir=dataset_dir,
    batch_size=32,
    num_workers=2
)

print(f'Target Classes ({len(class_names)}): {class_names}')
print(f'Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)} | Test Batches: {len(test_loader)}')



In [ ]:
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return tensor * std + mean

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for i, ax in enumerate(axes.flat):
    img = denormalize(images[i]).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(class_names[labels[i]].capitalize(), fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle('Batch of Augmented Training Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()



## 🧠 2. Pretrained Model Architecture

We utilize **MobileNetV2** pretrained on ImageNet-1K. Its inverted residual blocks and depthwise separable convolutions offer fast inference and parameter efficiency (~3.5M parameters).

The original 1000-class classifier is replaced with:
- `Dropout(p=0.3)`
- `Linear(in_features=1280, out_features=256)`
- `ReLU(inplace=True)`
- `Dropout(p=0.15)`
- `Linear(in_features=256, out_features=5)`



In [ ]:
checkpoint_path = os.path.join('..', 'models', 'best_model.pth')
model, checkpoint = load_checkpoint(checkpoint_path, device=device)

print(f"Architecture: {checkpoint.get('model_name')}")
print(f"Best Validation Accuracy: {checkpoint.get('val_accuracy', 0)*100:.2f}%")
print(f"Epoch: {checkpoint.get('epoch')}")
print('\nClassifier Head Architecture:\n', model.classifier)



## 📊 3. Training & Validation Curves

The model was trained in two stages:
1. **Warmup Phase (Epochs 1-4)**: Frozen backbone, AdamW optimizer ($lr=10^{-3}$).
2. **Fine-Tuning Phase (Epochs 5-7)**: Top convolutional blocks unfrozen, Cosine Annealing scheduler ($lr=10^{-4} \to 10^{-6}$).



In [ ]:
history_path = os.path.join('..', 'reports', 'training_history.json')
if os.path.exists(history_path):
    with open(history_path, 'r') as f:
        history = json.load(f)

    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Loss
    axes[0].plot(epochs, history['train_loss'], 'o-', label='Train Loss', color='#1f77b4', lw=2)
    axes[0].plot(epochs, history['val_loss'], 's--', label='Val Loss', color='#ff7f0e', lw=2)
    axes[0].set_title('Cross-Entropy Loss vs. Epochs', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend()

    # Accuracy
    axes[1].plot(epochs, [a * 100 for a in history['train_acc']], 'o-', label='Train Acc', color='#2ca02c', lw=2)
    axes[1].plot(epochs, [a * 100 for a in history['val_acc']], 's--', label='Val Acc', color='#d62728', lw=2)
    axes[1].set_title('Accuracy (%) vs. Epochs', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend()

    plt.tight_layout()
    plt.show()



## 🎯 4. Test Set Evaluation & Confusion Matrix

We evaluate the fine-tuned model against the completely unseen held-out test split (556 images).



In [ ]:
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc = (all_preds == all_labels).mean()
print(f'Test Accuracy: {test_acc * 100:.2f}%\n')
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 5.5))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=class_names, 
    yticklabels=class_names,
    annot_kws={'size': 12, 'weight': 'bold'}
)
plt.title(f'Confusion Matrix on Test Set (Accuracy: {test_acc*100:.2f}%)', fontsize=13, fontweight='bold', pad=10)
plt.xlabel('Predicted Class', fontsize=11)
plt.ylabel('Ground Truth Class', fontsize=11)
plt.tight_layout()
plt.show()



## 🌸 5. Inferencing on a New Image

Demonstration of loading a sample test image, preprocessing it, and computing top-k softmax probabilities.



In [ ]:
sample_file = os.path.join('..', 'sample_images', 'roses.jpg')
results = predict_image(sample_file, model_path=checkpoint_path)

img = Image.open(sample_file)
plt.figure(figsize=(4.5, 4.5))
plt.imshow(img)
plt.title(f'Prediction: {results[0][0].capitalize()} ({results[0][1]:.2f}%)', fontsize=13, fontweight='bold')
plt.axis('off')
plt.show()

print('Confidence Rankings:')
for rank, (cls, conf) in enumerate(results, 1):
    print(f' {rank}. {cls.capitalize():<12}: {conf:6.2f}%')

